# OFDM 系统可视化教程

本 Notebook 用 Python 从零实现一个完整 OFDM 系统，并展示每一步信号变化。

系统流程：

Bitstream → QPSK → Subcarrier Mapping → IFFT → Cyclic Prefix → Channel → FFT → Equalization → Demodulation


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False

np.random.seed(0)

## 1 系统参数

In [ ]:
N = 64
cp_len = 16
bits_per_symbol = 2

num_bits = N * bits_per_symbol
print('Number of bits:', num_bits)

## 2 生成比特流

In [ ]:
bits = np.random.randint(0,2,num_bits)

plt.figure(figsize=(10,2))
plt.step(range(len(bits)), bits, where="post")
plt.title('Input Bitstream')
plt.show()

## 3 QPSK 调制

In [ ]:
bit_pairs = bits.reshape((-1,2))

mapping = {
(0,0):1+1j,
(0,1):-1+1j,
(1,1):-1-1j,
(1,0):1-1j
}

symbols = np.array([mapping[tuple(b)] for b in bit_pairs])

In [ ]:
plt.scatter(symbols.real,symbols.imag)
plt.title('QPSK Constellation')
plt.grid()
plt.show()

## 4 子载波映射

In [ ]:
plt.stem(np.abs(symbols))
plt.title('Subcarrier Amplitude')
plt.show()

## 5 IFFT 生成 OFDM 信号

In [ ]:
time_signal = np.fft.ifft(symbols)

plt.plot(time_signal.real)
plt.title('OFDM Time Signal')
plt.show()

## 6 Cyclic Prefix

In [ ]:
cp = time_signal[-cp_len:]

tx_signal = np.concatenate([cp,time_signal])

plt.plot(tx_signal.real)
plt.axvline(cp_len,color='red')
plt.title('OFDM with CP')
plt.show()

## 7 OFDM 频谱

In [ ]:
spectrum = np.fft.fftshift(np.fft.fft(tx_signal,1024))

plt.plot(20*np.log10(np.abs(spectrum)))
plt.title('OFDM Spectrum')
plt.show()

## 8 PAPR

In [ ]:
papr = np.max(np.abs(tx_signal)**2)/np.mean(np.abs(tx_signal)**2)
print('PAPR:', papr)

## 9 AWGN 信道

In [ ]:
snr_db = 20

signal_power = np.mean(np.abs(tx_signal)**2)
noise_power = signal_power/(10**(snr_db/10))

noise = np.sqrt(noise_power/2)*(np.random.randn(len(tx_signal)) + 1j*np.random.randn(len(tx_signal)))

rx_signal = tx_signal + noise

## 10 FFT 接收

In [ ]:
rx_no_cp = rx_signal[cp_len:]
rx_symbols = np.fft.fft(rx_no_cp)

plt.scatter(rx_symbols.real,rx_symbols.imag)
plt.title('Received Constellation')
plt.grid()
plt.show()

## 11 多径信道

In [ ]:
h = np.array([1,0.5,0.3])

rx_multipath = np.convolve(tx_signal,h)

## 12 信道频率响应

In [ ]:
H = np.fft.fft(h,N)

plt.plot(np.abs(H))
plt.title('Channel Frequency Response')
plt.show()

## 13 信道均衡

In [ ]:
Y = np.fft.fft(rx_multipath[cp_len:cp_len+N])

X_eq = Y/(H+1e-8)

plt.scatter(X_eq.real,X_eq.imag)
plt.title('Equalized Constellation')
plt.grid()
plt.show()

## 14 OFDM Resource Grid

In [ ]:
num_sc = 64
num_sym = 14

grid = np.random.randn(num_sc,num_sym)

plt.imshow(grid,aspect='auto')
plt.title('OFDM Resource Grid')
plt.xlabel('OFDM Symbol')
plt.ylabel('Subcarrier')
plt.show()